# Week 5: CNNs For Text

This notebook studies `TextCNN` for sentiment classification using the same IMDb tokenization pipeline, split seed, and pretrained `DAN` embedding used in Weeks 3 and 4.

## Learning Goals
- Understand `1D` convolution as a sliding window over token embeddings.
- Show how multiple filters at widths `2`, `3`, and `4` learn different local phrase detectors.
- Explain why parameter sharing lets one filter detect a useful pattern anywhere in the sequence.
- Compare global max pooling and mean pooling conceptually.
- Build and train a `TextCNN` model on IMDb.
- Inspect learned feature maps after training and connect them back to pooling.
- Close with a short CNN vs. RNN reflection.

## Reused Assets
This week reuses the saved artifacts from previous weeks so the modeling comparison stays consistent:

- `./data/imdb_bpe_dataset.pt`
- `./data/bpe_vocab.json`
- `./data/bpe_merges.json`
- `./models/vocab_config.json`
- `./models/dan_imdb.pt`

The dataset split follows the Week 4 setup: `80/10/10` with `manual_seed(204)`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from utils import (
    bpe_batch_encode,
    build_conv_mask,
    clean_memory,
    evaluate_model,
    get_device,
    load_json,
    load_pretrained_embedding,
    load_torch_dataset,
    plot_feature_map_heatmap,
    train_test_model,
    word_tokenizer,
)

In [ ]:
paths = {
    "dataset": Path("./data/imdb_bpe_dataset.pt"),
    "bpe_vocab": Path("./data/bpe_vocab.json"),
    "bpe_merges": Path("./data/bpe_merges.json"),
    "vocab_config": Path("./models/vocab_config.json"),
    "dan_state": Path("./models/dan_imdb.pt"),
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required Week 5 assets:\n" + "\n".join(missing))

input_ids, labels, attention_masks = load_torch_dataset(str(paths["dataset"]))
bpe_vocab = load_json(str(paths["bpe_vocab"]))
bpe_merges = load_json(str(paths["bpe_merges"]))
vocab_config = load_json(str(paths["vocab_config"]))

pad_id = vocab_config["pad_id"]
vocab_size = vocab_config["vocab_size"]
max_seq = vocab_config["max_seq"]
pad_token = vocab_config["pad_token"]
device = get_device()

print(f"input_ids shape: {tuple(input_ids.shape)}")
print(f"labels shape: {tuple(labels.shape)}")
print(f"attention mask shape: {tuple(attention_masks.shape)}")
print(f"device: {device}")

## 1D CNN Intuition
A `TextCNN` treats text as a sequence of token embeddings. A convolution with kernel size `2` looks at adjacent 2-token windows, kernel size `3` looks at 3-token windows, and so on.

Each filter is its own learned pattern detector. If we use `64` filters for width `2`, that means the model learns `64` different bigram-style detectors rather than one generic bigram feature.

The same filter is reused across the full sequence. That is parameter sharing, and it means a useful phrase pattern can be detected anywhere in the review.

## Pooling Intuition
Each filter produces a feature map: one score for every valid sliding-window position.

- `max pooling` asks: did this pattern appear strongly anywhere?
- `mean pooling` asks: how active was this pattern on average?

For sentiment classification, max pooling is often a strong choice because one phrase such as `absolutely terrible` or `really loved it` can be enough to carry the label.

In [ ]:
example_scores = torch.tensor([0.1, 0.2, 2.8, 0.3])
print("example feature map:", example_scores.tolist())
print("max pooling keeps:", float(example_scores.max()))
print("mean pooling keeps:", float(example_scores.mean()))

In [ ]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        padding_idx: int,
        num_filters: int,
        kernel_sizes=(2, 3, 4),
        dropout: float = 0.3,
        num_classes: int = 1,
        pool_mode: str = "max",
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.kernel_sizes = tuple(kernel_sizes)
        self.pool_mode = pool_mode
        self.dropout = nn.Dropout(dropout)
        self.convs = nn.ModuleList(
            [nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in self.kernel_sizes]
        )
        self.linear = nn.Linear(num_filters * len(self.kernel_sizes), num_classes)

    def _pool_feature_map(self, feature_map: torch.Tensor, conv_mask: torch.Tensor) -> torch.Tensor:
        mask = conv_mask.unsqueeze(1)
        if self.pool_mode == "max":
            masked_map = feature_map.masked_fill(~mask, -1e9)
            return masked_map.max(dim=-1).values
        if self.pool_mode == "mean":
            masked_map = feature_map * mask.float()
            denom = mask.float().sum(dim=-1).clamp(min=1.0)
            return masked_map.sum(dim=-1) / denom
        raise ValueError(f"Unsupported pool_mode: {self.pool_mode}")

    def extract_feature_maps(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)
        conv_input = embedded.transpose(1, 2)

        feature_maps = {}
        pooled_outputs = []
        for kernel_size, conv in zip(self.kernel_sizes, self.convs):
            conv_out = conv(conv_input)
            activated = torch.relu(conv_out)
            conv_mask = build_conv_mask(attention_mask, kernel_size)
            pooled = self._pool_feature_map(activated, conv_mask)
            feature_maps[kernel_size] = {
                "activations": activated,
                "mask": conv_mask,
                "pooled": pooled,
            }
            pooled_outputs.append(pooled)

        sentence_features = torch.cat(pooled_outputs, dim=1)
        sentence_features = self.dropout(sentence_features)
        return feature_maps, sentence_features

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        _, sentence_features = self.extract_feature_maps(input_ids, attention_mask)
        return self.linear(sentence_features)

In [ ]:
dataset = TensorDataset(input_ids, attention_masks, labels.float())
n = len(dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val
g = torch.Generator().manual_seed(204)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

print(f"train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

In [ ]:
embedding_dim = 384
num_filters = 64
kernel_sizes = (2, 3, 4)
dropout = 0.3
pool_mode = "max"

model = TextCNN(
    vocab_size=vocab_size,
    embed_dim=embedding_dim,
    padding_idx=pad_id,
    num_filters=num_filters,
    kernel_sizes=kernel_sizes,
    dropout=dropout,
    num_classes=1,
    pool_mode=pool_mode,
).to(device)

load_pretrained_embedding(model, str(paths["dan_state"]))
model = model.to(device)
print("Initialized embedding layer from Week 3 DAN weights.")

## Train TextCNN
The embedding layer starts from the saved `DAN` weights but remains trainable. This keeps the setup consistent with earlier weeks while still letting the CNN adapt the embedding space for local phrase detection.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
patience = 4
num_epochs = 12
lr = 2e-3
weight_decay = 1e-4
save_path = "./models/final_textcnn_imdb.pt"

best_model = train_test_model(
    device=device,
    model=model,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    num_epochs=num_epochs,
    lr=lr,
    weight_decay=weight_decay,
    is_binary=True,
    save_path=save_path,
)

In [ ]:
test_loss, test_acc = evaluate_model(
    model=best_model,
    data_loader=test_loader,
    criterion=criterion,
    device=device,
    is_binary=True,
)
print(f"test_loss={test_loss:.4f} | test_acc={test_acc:.4f}")

## Challenge Set
This small handwritten set is not a benchmark. It is a behavioral probe aimed at local phrase effects that a sliding-window model should care about, including negation, intensifiers, and phrase relocation.

In [ ]:
challenge_set = {
    "negation": [
        {"text": "This movie is not good.", "label": 0},
        {"text": "This movie is not bad.", "label": 1},
        {"text": "I did not enjoy this film.", "label": 0},
        {"text": "I did not hate this film.", "label": 1},
    ],
    "intensifiers": [
        {"text": "This movie is absolutely wonderful.", "label": 1},
        {"text": "This movie is really disappointing.", "label": 0},
        {"text": "The ending was incredibly moving.", "label": 1},
        {"text": "The ending was deeply frustrating.", "label": 0},
    ],
    "phrase_relocation": [
        {"text": "Absolutely wonderful acting saves this movie.", "label": 1},
        {"text": "This movie is saved by absolutely wonderful acting.", "label": 1},
        {"text": "Really disappointing writing ruins this movie.", "label": 0},
        {"text": "This movie is ruined by really disappointing writing.", "label": 0},
    ],
    "contrast": [
        {"text": "The opening is weak but the ending is fantastic.", "label": 1},
        {"text": "The opening is fantastic but the ending is weak.", "label": 0},
    ],
}

all_sentences = [sample["text"] for group in challenge_set.values() for sample in group]
all_labels = torch.tensor([sample["label"] for group in challenge_set.values() for sample in group], dtype=torch.float32)
all_groups = [group_name for group_name, items in challenge_set.items() for _ in items]

challenge_ids, challenge_masks = bpe_batch_encode(
    sentences=all_sentences,
    bpe_merges=bpe_merges,
    bpe_vocab=bpe_vocab,
    max_seq=max_seq,
    pad_token=pad_token,
)

challenge_ids = challenge_ids.to(device)
challenge_masks = challenge_masks.to(device)
all_labels = all_labels.to(device)

In [ ]:
best_model.eval()
with torch.no_grad():
    logits = best_model(challenge_ids, challenge_masks)
    probs = torch.sigmoid(logits).squeeze(-1)
    preds = (probs > 0.5).float()

for group_name, text, label, prob, pred in zip(all_groups, all_sentences, all_labels.tolist(), probs.tolist(), preds.tolist()):
    print(f"[{group_name}] label={int(label)} pred={int(pred)} prob={prob:.3f} | {text}")

## Inspect Learned Feature Maps
Now that the model has trained filters, the feature maps are much easier to interpret than they would be at random initialization.

Because this notebook reuses the byte-level BPE pipeline from earlier weeks, the x-axis below is shown as sliding-window position rather than clean word bigrams or trigrams.

In [ ]:
probe_text = "The ending is absolutely wonderful even if the opening is weak."
probe_ids, probe_masks = bpe_batch_encode(
    sentences=[probe_text],
    bpe_merges=bpe_merges,
    bpe_vocab=bpe_vocab,
    max_seq=max_seq,
    pad_token=pad_token,
)
probe_ids = probe_ids.to(device)
probe_masks = probe_masks.to(device)

best_model.eval()
with torch.no_grad():
    feature_maps, sentence_features = best_model.extract_feature_maps(probe_ids, probe_masks)
    prob = torch.sigmoid(best_model(probe_ids, probe_masks)).item()

print(f"probe text: {probe_text}")
print(f"word tokens: {word_tokenizer(probe_text)}")
print(f"predicted positive probability: {prob:.3f}")

for kernel_size, payload in feature_maps.items():
    activations = payload["activations"][0]
    pooled = payload["pooled"][0]
    print(f"kernel={kernel_size} | feature_map shape={tuple(activations.shape)} | pooled shape={tuple(pooled.shape)}")
    plot_feature_map_heatmap(
        activations,
        kernel_size=kernel_size,
        title=f"Kernel size {kernel_size} feature map for probe sentence",
        max_filters=12,
    )

In [ ]:
max_model = best_model
mean_model = TextCNN(
    vocab_size=vocab_size,
    embed_dim=embedding_dim,
    padding_idx=pad_id,
    num_filters=num_filters,
    kernel_sizes=kernel_sizes,
    dropout=dropout,
    num_classes=1,
    pool_mode="mean",
).to(device)
mean_model.load_state_dict(max_model.state_dict(), strict=False)

with torch.no_grad():
    _, max_features = max_model.extract_feature_maps(probe_ids, probe_masks)
    _, mean_features = mean_model.extract_feature_maps(probe_ids, probe_masks)

print("first 12 pooled features with max pooling:")
print(max_features[0, :12].detach().cpu())
print("first 12 pooled features with mean pooling:")
print(mean_features[0, :12].detach().cpu())

## CNN vs. RNN
- CNNs are fast and parallelizable because all windows can be processed at once.
- CNNs are especially good when the label depends on local phrase evidence such as negation or strong sentiment expressions.
- RNNs are more naturally aligned with information that accumulates gradually across a long sequence.
- Gradient clipping is often more important for RNNs because recurrence can make optimization less stable; we keep it here as a general safeguard, not because it is a defining part of CNN training.
- In practice, `TextCNN` is a very strong model when local phrase detection is the main learning problem.

## Summary
`TextCNN` replaces hand-written n-gram counting with learned sliding-window detectors over embeddings. Multiple filters learn different local patterns, parameter sharing makes those detectors reusable across positions, and global max pooling turns strong local evidence into a sentence-level signal for classification.